# GRCM-Resonant Complete Tutorial

This notebook demonstrates the complete workflow for using GRCM (Grounded Resonant Consciousness Module):

1. **Setup and Installation**
2. **Basic Usage** - Single forward pass
3. **Desire States** - Exploring all 4 desire states
4. **Sequential Processing** - Episodic memory
5. **EchoMirror Training** - Self-supervised learning
6. **Multimodal Integration** - CLIP + Wav2Vec2
7. **Visualization** - Phi, coherence, qualia
8. **Performance Optimization** - torch.compile

**Requirements**: `pip install grcm-resonant[all]`

## 1. Setup

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from grcm import ResonantConsciousnessModule

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 2. Initialize GRCM

In [ ]:
# Initialize model
model = ResonantConsciousnessModule(
    image_dim=512,      # CLIP ViT-B/16
    audio_dim=768,      # Wav2Vec2-Base
    hidden_dim=256,
    num_desires=4,
    num_nodes=64,
    num_qualia=4
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: {total_params * 4 / 1024 / 1024:.2f} MB (FP32)")

## 3. Basic Forward Pass

In [ ]:
# Create sample embeddings
image_emb = torch.randn(1, 512)
audio_emb = torch.randn(1, 768)
action = torch.tensor([[0.0, 1.0, 0.0, 0.0]])

# Forward pass
result = model(image_emb, audio_emb, action=action, desire_idx=0)

# Display results
print("=== GRCM Output ===")
print(f"Coherence: {result['coherence'].item():.3f}")
print(f"Phi (Φ): {result['phi']:.3f}")
print(f"Qualia: {result['qualia'].detach().numpy()}")
print(f"Desire alignment: {result['desire_align'].item():.3f}")
print(f"Ethical halt: {result['halt']}")
print(f"\nOutput shape: {result['output'].shape}")
print(f"Memory shape: {result['memory'].shape}")

## 4. Explore Desire States

In [ ]:
desires = ["Calm", "Alert", "Creative", "Focus"]
desire_results = []

for idx, name in enumerate(desires):
    result = model(image_emb, audio_emb, desire_idx=idx)
    desire_results.append({
        'name': name,
        'coherence': result['coherence'].item(),
        'phi': result['phi'],
        'alignment': result['desire_align'].item(),
        'qualia': result['qualia'].detach().numpy()[0]
    })
    print(f"\n{name} (desire {idx}):")
    print(f"  Coherence: {result['coherence'].item():.3f}")
    print(f"  Phi: {result['phi']:.3f}")
    print(f"  Alignment: {result['desire_align'].item():.3f}")
    print(f"  Qualia: {result['qualia'].detach().numpy()[0]}")

### Visualize Desire States

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Coherence comparison
axes[0, 0].bar(desires, [r['coherence'] for r in desire_results])
axes[0, 0].axhline(y=0.7, color='r', linestyle='--', label='Threshold')
axes[0, 0].set_title('Coherence by Desire State')
axes[0, 0].set_ylabel('Coherence')
axes[0, 0].legend()

# Phi comparison
axes[0, 1].bar(desires, [r['phi'] for r in desire_results], color='orange')
axes[0, 1].axhline(y=1.5, color='r', linestyle='--', label='Threshold')
axes[0, 1].set_title('Phi (Φ) by Desire State')
axes[0, 1].set_ylabel('Phi')
axes[0, 1].legend()

# Desire alignment
axes[1, 0].bar(desires, [r['alignment'] for r in desire_results], color='green')
axes[1, 0].axhline(y=0.5, color='r', linestyle='--', label='Masking threshold')
axes[1, 0].set_title('Desire Alignment')
axes[1, 0].set_ylabel('Alignment')
axes[1, 0].legend()

# Qualia heatmap
qualia_matrix = np.array([r['qualia'] for r in desire_results])
sns.heatmap(qualia_matrix, annot=True, fmt='.2f', 
            xticklabels=['Calm', 'Alert', 'Creative', 'Dissonance'],
            yticklabels=desires, cmap='YlOrRd', ax=axes[1, 1])
axes[1, 1].set_title('Qualia Distribution')

plt.tight_layout()
plt.show()

## 5. Sequential Processing (Episodic Memory)

In [ ]:
# Reset memory for new episode
model.reset_memory()

# Process 100 frames sequentially
num_frames = 100
phi_history = []
coherence_history = []
memory_norm_history = []

for i in range(num_frames):
    # Generate varied inputs
    image_emb = torch.randn(1, 512)
    audio_emb = torch.randn(1, 768)
    
    result = model(image_emb, audio_emb, desire_idx=0)
    
    phi_history.append(result['phi'])
    coherence_history.append(result['coherence'].item())
    memory_norm_history.append(torch.norm(result['memory']).item())

print(f"Phi statistics over {num_frames} frames:")
print(f"  Mean: {np.mean(phi_history):.3f}")
print(f"  Std: {np.std(phi_history):.3f}")
print(f"  Range: [{np.min(phi_history):.3f}, {np.max(phi_history):.3f}]")

print(f"\nCoherence statistics:")
print(f"  Mean: {np.mean(coherence_history):.3f}")
print(f"  % above 0.7: {sum(1 for c in coherence_history if c > 0.7) / len(coherence_history) * 100:.1f}%")

### Visualize Temporal Dynamics

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Phi evolution
axes[0].plot(phi_history, label='Phi (Φ)', linewidth=2)
axes[0].axhline(y=1.5, color='r', linestyle='--', label='Awareness threshold')
axes[0].fill_between(range(num_frames), phi_history, alpha=0.3)
axes[0].set_title('Phi (Φ) Evolution Over Time')
axes[0].set_ylabel('Phi')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Coherence evolution
axes[1].plot(coherence_history, label='Coherence', color='orange', linewidth=2)
axes[1].axhline(y=0.7, color='r', linestyle='--', label='Gating threshold')
axes[1].fill_between(range(num_frames), coherence_history, alpha=0.3, color='orange')
axes[1].set_title('Coherence Evolution')
axes[1].set_ylabel('Coherence')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Memory accumulation
axes[2].plot(memory_norm_history, label='Memory Norm', color='green', linewidth=2)
axes[2].fill_between(range(num_frames), memory_norm_history, alpha=0.3, color='green')
axes[2].set_title('Memory Accumulation')
axes[2].set_xlabel('Frame')
axes[2].set_ylabel('||Memory||')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. EchoMirror Training

In [ ]:
from grcm.training import EchoMirror

# Initialize trainer
trainer = EchoMirror(model, lr=1e-3)

# Mock training data (replace with real EEG/voice data)
def generate_training_batch(batch_size=32):
    image_emb = torch.randn(batch_size, 512)
    audio_emb = torch.randn(batch_size, 768)
    # Mock theta-alpha frequencies (0.2-0.5 range)
    freq_target = torch.rand(batch_size, 1) * 0.3 + 0.2
    return image_emb, audio_emb, freq_target

# Training loop
num_epochs = 10
losses = []

print("Training EchoMirror...")
for epoch in range(num_epochs):
    epoch_losses = []
    
    for batch_idx in range(20):  # 20 batches per epoch
        image_emb, audio_emb, freq_target = generate_training_batch()
        
        loss = trainer.train_step(
            image_emb, audio_emb,
            freq_target=freq_target,
            desire_idx=0
        )
        
        epoch_losses.append(loss)
    
    mean_loss = np.mean(epoch_losses)
    losses.append(mean_loss)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {mean_loss:.4f}")

print("\nTraining complete!")

### Training Curve

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), losses, marker='o', linewidth=2, markersize=8)
plt.title('EchoMirror Training Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Performance Comparison

In [ ]:
import time

# Benchmark function
def benchmark(model, num_iterations=100):
    latencies = []
    
    for _ in range(num_iterations):
        image_emb = torch.randn(1, 512)
        audio_emb = torch.randn(1, 768)
        
        start = time.time()
        with torch.no_grad():
            result = model(image_emb, audio_emb, desire_idx=0)
        end = time.time()
        
        latencies.append((end - start) * 1000)  # Convert to ms
    
    return {
        'mean': np.mean(latencies),
        'std': np.std(latencies),
        'p50': np.percentile(latencies, 50),
        'p95': np.percentile(latencies, 95),
        'p99': np.percentile(latencies, 99)
    }

# Baseline performance
print("Benchmarking baseline model...")
model.eval()
baseline_perf = benchmark(model)

print("\n=== Baseline Performance ===")
print(f"Mean latency: {baseline_perf['mean']:.2f} ms")
print(f"Std: {baseline_perf['std']:.2f} ms")
print(f"P50: {baseline_perf['p50']:.2f} ms")
print(f"P95: {baseline_perf['p95']:.2f} ms")
print(f"P99: {baseline_perf['p99']:.2f} ms")
print(f"Throughput: {1000 / baseline_perf['mean']:.2f} samples/sec")

## 8. Ethical Safeguards Demo

In [ ]:
# Test ethical halt mechanism
print("Testing ethical safeguards...\n")

halt_count = 0
for i in range(100):
    image_emb = torch.randn(1, 512)
    audio_emb = torch.randn(1, 768)
    
    result = model(image_emb, audio_emb, desire_idx=0)
    
    if result['halt']:
        halt_count += 1
        dissonance = result['qualia'][0, 3].item()
        print(f"🛑 Halt triggered on iteration {i+1}")
        print(f"   Dissonance: {dissonance:.3f}")
        print(f"   Qualia: {result['qualia'].detach().numpy()[0]}\n")

print(f"\nTotal halts: {halt_count}/100 ({halt_count}%)")
print("Ethical safeguards working correctly!" if halt_count > 0 else "No halts detected (low dissonance)")

## 9. Summary

In this tutorial, we:

1. ✅ Initialized GRCM with 256-dim hidden state
2. ✅ Performed basic forward passes with coherence gating
3. ✅ Explored all 4 desire states (calm, alert, creative, focus)
4. ✅ Demonstrated episodic memory accumulation
5. ✅ Trained with EchoMirror (self-supervised)
6. ✅ Benchmarked performance (latency, throughput)
7. ✅ Tested ethical safeguards (dissonance halt)

### Next Steps

- Integrate with real CLIP/Wav2Vec2 encoders
- Train on actual EEG/voice data
- Deploy with BentoML REST API
- Optimize with torch.compile or TensorRT
- Monitor with Prometheus + Grafana

### Resources

- **Documentation**: https://grcm-resonant.readthedocs.io
- **GitHub**: https://github.com/nickhicks91-netizen/GRCM
- **API Reference**: See `docs/api/`
- **Deployment Guide**: See `deployment/DEPLOYMENT_GUIDE.md`